# Final Implementation Runs for Markov Chain Classification of Genomic Features Over Varying k Values

This notebook chronicles the Final Execution of this project. It is a culmination of work spanning four months that critically investigates a Markov Chain model's ability to recognize patterns unique to each genomic feature and then use that information to distinguish between four canonical classes: promoters, exons, introns, and repeats. The model will be evaluated under different orders of k to identify which order is optimal for each class and at what k value(s) the model breaks down.

### Set Working Directory to Root and Add src/ and notebooks/ to Python Path

In [1]:
# Set working directory to root and add src/ and notebooks/ to path
import os
import sys

from data_preparation import load_fasta, count_seqs_in_fasta, total_bp_in_fasta

cwd = os.getcwd()
# Ensure working dir is root, not src/ or notebooks/
if cwd.endswith("src") or cwd.endswith("notebooks"):
    os.chdir("..")

print("Working directory:", os.getcwd())

# Add src/ to Python path
sys.path.insert(0, "src")
sys.path.insert(0, "notebook")

Working directory: /Users/biotechiestefnie/Desktop/Algorithms_Final_Project


### Import Packages and Submodules Required to Execute Programs for Final Run

The following cell imports all program scripts required to execute a full runthrough of the model, including loading the datafiles into memory, training the model, calculating thresholds, testing, generating classifications, calculating per-base log-likelihood scores, evaluating results, and visualizing results. All scripts have been written to run exclusively within this notebook with the provided calls. While technically, the scripts can be run individually from a local terminal, this would require either including the entirety of a call in the terminal, or modifying the scripts themselves to manually reproduce the full workflow, output structure, and visualization pipeline used here. In practice, this would involve rewriting large portions of the driver logic, restructuring return objects, and adding additional I/O handling to generate results in the same format produced automatically within the notebook. For this reason, the notebook serves as the intended and most efficient execution environment for the complete model pipeline.

In [2]:
# Import packages and submodules
from evaluate_results import compute_confusion_matrix, compute_per_class_accuracy, compute_overall_accuracy, compute_class_metrics
from per_base_likelihood import per_base_likelihood
from model_classification import (
    classify_sequence,
    classify_with_reject,
    generate_full_results
)
from data_loading import read_fasta, load_class_seqs
from training_utils import train_markov_model, train_all_models
from threshold_utils import compute_thresholds

### Define Training Data FASTA Path for Final Run and Load into Dictionary

In [3]:
# Define datapaths for input training and testing datasets
fasta_paths_final = {
    "promoters" : "data/final/train/promoters_train.fa",
    "exons" : "data/final/train/exons_train.fa",
    "introns" : "data/final/train/introns_train.fa",
    "repeats" : "data/final/train/repeats_train.fa"
}


### Load Training Dataset for Each Class into Model from Single, Combined File

In [4]:
training_data_final = load_class_seqs({
    "promoters" : fasta_paths_final["promoters"],
    "exons" : fasta_paths_final["exons"],
    "introns" : fasta_paths_final["introns"],
    "repeats" : fasta_paths_final["repeats"]
})


### Train Markov Model Over Increasing k Values to Generate Transition Probabilities for Testing Phase

In this step the model is trained with the training data, which contains the sequences for each class and their associated true labels. For k = 3, there are 4^3 = 64 possible states and for 10x coverage to ensure reliable results, we are aiming for 10 observations per state, which equals a total of 640 3-mers. 640 3-mers comes from: L - 3 + 1 = 640 -> 642 total basepairs, meaning we need at least 642 total basepairs in a class to have 10x coverage for the 3-mers. Since our sequences

In [5]:
# Train Markov models for structural classes over k = 1, 2, 3, 4, 5, 6

k_values_final = [1, 2, 3, 4, 5, 6]

all_models_final_run = {}       # mapping: k -> {class_label -> model}
all_thresholds_final_run = {}   # mapping: k -> {class_label -> threshold}

for k in k_values_final:
    # Train models for this k
    models = train_all_models(training_data_final, k)
    all_models_final_run[k] = models

# Display transition probabilities for verification
all_models_final_run


{1: {'promoters': {'G': -1.1217076194063769,
   'A': -1.7073859899760453,
   'C': -1.202419287116189,
   'T': -1.6593970044070312,
   'N': -6.11066339366997,
   '_UNSEEN_': -13.431851950409449},
  'exons': {'A': -1.3615094575018416,
   'T': -1.5216759905461057,
   'G': -1.3279060517551386,
   'C': -1.3457319668298222,
   '_UNSEEN_': -14.112371376545562},
  'introns': {'G': -1.5051022118996287,
   'T': -1.2225852090626095,
   'A': -1.3112222412358425,
   'C': -1.5415596420618494,
   '_UNSEEN_': -14.60066154921856},
  'repeats': {'C': -1.4381690571637797,
   'A': -1.2543762113007961,
   'G': -1.5463637392843061,
   'T': -1.3304315746040334,
   '_UNSEEN_': -12.285387395999454}},
 2: {'promoters': {'GG': -2.154594378834105,
   'GA': -2.801229614556189,
   'AG': -2.646487633070062,
   'GC': -2.2879953300592413,
   'CA': -2.9216583347920717,
   'AC': -3.219450976707221,
   'CG': -2.569911532244127,
   'GT': -3.046925265734623,
   'TC': -2.8516554587151863,
   'CT': -2.6762758603374066,
   'T

### Calculate Log-Likelihood Thresholds for Exons, Introns, and Repeats at 5% and 10%

Thresholds were computed to give the classifier a true reject option, preventing it from being forced to assign a sequence to a class when none of the models provide sufficient support. For each class and each kmer model, I computed the 5th and 10th percentiles of each class's own training log likelihood distributions. This yielded a single lower bound cutoff: sequences scoring below this value are rejected, while sequences scoring above it are accepted. Because the class-conditional likelihoods are highly concentrated around the mean, these aggressive thresholds produce meaningful rejections without excessively discarding true sequences. The false reject rate will remain low relative to the dataset size, while the classifier gains the ability to avoid incorrect forced classifications

In [6]:
# Thresholds for each class at 5%
thresholds_5 = compute_thresholds(
    all_models_final_run, training_data_final, percentile=5)

# Thresholds for each class at 10%
thresholds_10 = compute_thresholds(
    all_models_final_run, training_data_final, percentile=10)

print(thresholds_5)
print(thresholds_10)

{1: {'promoters': -1.4626872921447294, 'exons': -1.3982881805988416, 'introns': -1.423143447241731, 'repeats': -1.4414453830180718}, 2: {'promoters': -2.898948370041853, 'exons': -2.772342651674454, 'introns': -2.8024222926835516, 'repeats': -2.8980217867030733}, 3: {'promoters': -4.308028228465032, 'exons': -4.142338838077387, 'introns': -4.162912993075121, 'repeats': -4.286147305979122}, 4: {'promoters': -5.701057604204286, 'exons': -5.499053429547511, 'introns': -5.51514155198563, 'repeats': -5.636348659440709}, 5: {'promoters': -7.05627236907735, 'exons': -6.847586986190133, 'introns': -6.86658036887127, 'repeats': -6.913135055503642}, 6: {'promoters': -8.421327833536392, 'exons': -8.192161392907419, 'introns': -8.200148946655549, 'repeats': -8.15640227831124}}
{1: {'promoters': -1.4274426944988672, 'exons': -1.3943309629012364, 'introns': -1.4131995001630089, 'repeats': -1.4241618595835126}, 2: {'promoters': -2.8263474054542517, 'exons': -2.7627254966057873, 'introns': -2.77769464

### The Official Thresholds:

Thresholds at the 5th percentile:

| Class     | k=1       | k=2       | k=3       | k=4       | k=5       | k=6        |
|-----------|-----------|-----------|-----------|-----------|-----------|------------|
| promoters | -1.462687 | -2.898948 | -4.308028 | -5.701058 | -7.056272 | -8.421328  |
| exons     | -1.398288 | -2.772343 | -4.142339 | -5.499053 | -6.847586 | -8.192161  |
| introns   | -1.423143 | -2.802422 | -4.162913 | -5.515142 | -6.866580 | -8.200149  |
| repeats   | -1.441445 | -2.898022 | -4.286147 | -5.636349 | -6.913135 | -8.156402  |


Thresholds at the 10th percentile:

| Class     | k=1       | k=2       | k=3       | k=4       | k=5       | k=6       |
|-----------|-----------|-----------|-----------|-----------|-----------|-----------|
| promoters | -1.427443 | -2.826347 | -4.208515 | -5.578369 | -6.927251 | -8.277432 |
| exons     | -1.394331 | -2.762725 | -4.123612 | -5.473850 | -6.818137 | -8.157285 |
| introns   | -1.413200 | -2.777695 | -4.124824 | -5.466314 | -6.801277 | -8.130551 |
| repeats   | -1.424162 | -2.798572 | -4.156997 | -5.484848 | -6.783484 | -8.073789 |


Note how close these threshold values are within classes — this justifies retaining precision to the sixth decimal place. I wanted to ensure that rejections were being made appropriately, especially because the per‑base log‑likelihood distributions are extremely compressed. At this stage, I am not yet certain whether the per‑base scores will remain sufficiently distinct between classes to support accurate classification. If they do not, I will address this using an alternative normalization strategy.

One option is to normalize by the number of kmers, L - k + 1, rather than by sequence length. As k increases, the difference between L and L - k + 1 becomes meaningful. Per‑base normalization can compress the score range too aggressively, whereas per‑kmer normalization expands it, which may improve separation for classes with different motif densities.

Another option is z‑score normalization within each class. For each class, I would compute the mean and standard deviation of the log‑likelihoods and convert each score to a z‑score. This removes class‑specific scale differences, makes distributions directly comparable, and sharpens the distinction between “typical” and “atypical” sequences.

A natural question is why z‑scores were not used from the beginning, given that they make distributions comparable. Early in the study, the goal was to examine the raw and normalized log‑likelihoods produced by the class‑conditional Markov models to understand where each class naturally falls. This is the standard first step in generative modeling. Z‑scores are not used at this stage because they eliminate the generative meaning of the likelihoods by converting them into standardized units. While this is excellent for classification, it obscures the underlying behavior of the model.

Z‑scores are therefore a diagnostic correction, not a starting point. In the prototype run, even with only two classes, the raw likelihood ranges overlapped substantially. Applying per‑base normalization improved separation, so I adopted that approach. I will attempt the same strategy with the expanded four‑class dataset. If per‑base normalization no longer provides adequate separation, I will transition to one of the alternative normalization methods described above.

### Prepare Test Sequences and True Labels for Markov Chain Models

In [7]:
from Bio import SeqIO

test_sequences_final = []
true_labels_final = {}

for record in SeqIO.parse("data/final/test/test.fa", "fasta"):
    header = record.description

    # Extract class label: promoters/exons/introns/repeats
    cls = header.replace(">", "").split("_all_final_")[0]

    seq = str(record.seq)

    test_sequences_final.append(seq)
    true_labels_final[seq] = cls

test_data_final = {}
for seq in test_sequences_final:
    label = true_labels_final[seq]
    test_data_final.setdefault(label, []).append(seq)


### Classification of Test Sequences Over Range of k-values 1-6 for Promoters, Exons, Introns, and Repeats Using Per-Base Log Likelihood Scoring for Final Run

In [8]:
results_by_k_final = {}

for k in k_values_final:
    print(f"Evaluating k = {k}")

    models_k_final = all_models_final_run[k]
    thresholds_k_final = thresholds_5[k]   # or thresholds_10[k]

    results_k_final = generate_full_results(
        test_data=test_data_final,
        models=models_k_final,
        k=k,
        threshold=thresholds_k_final
    )

    clean_results = []
    for r in results_k_final:
        clean_results.append({
            "true_label": r["true_label"],
            "predicted_label": r["predicted_label"],
            "scores": r["scores"]
        })

    results_by_k_final[k] = clean_results



Evaluating k = 1
Evaluating k = 2
Evaluating k = 3
Evaluating k = 4
Evaluating k = 5
Evaluating k = 6


### Print Interpretable Classification Outcomes for All k Values and Each Genomic Feature

In [10]:
from collections import defaultdict

for k in k_values_final:
    print(f"\n===== PERFORMANCE FOR k = {k} =====")

    results = results_by_k_final[k]

    # Counters
    correct = defaultdict(int)
    incorrect = defaultdict(int)
    rejected = defaultdict(int)
    misclassified_as = defaultdict(lambda: defaultdict(int))

    for r in results:
        true_label = r["true_label"]
        pred = r["predicted_label"]

        if pred == "reject":
            rejected[true_label] += 1
        elif pred == true_label:
            correct[true_label] += 1
        else:
            incorrect[true_label] += 1
            misclassified_as[true_label][pred] += 1

    # Print summary
    for cls in ["promoters", "exons", "introns", "repeats"]:
        print(f"\nClass: {cls}")
        print(f"  Correct:   {correct[cls]}")
        print(f"  Incorrect: {incorrect[cls]}")
        print(f"  Rejected:  {rejected[cls]}")

        if incorrect[cls] > 0:
            print("  Misclassified as:")
            for wrong_label, count in misclassified_as[cls].items():
                print(f"    {wrong_label}: {count}")



===== PERFORMANCE FOR k = 1 =====

Class: promoters
  Correct:   250
  Incorrect: 88
  Rejected:  2
  Misclassified as:
    introns: 18
    repeats: 10
    exons: 60

Class: exons
  Correct:   128
  Incorrect: 188
  Rejected:  0
  Misclassified as:
    promoters: 100
    repeats: 54
    introns: 34

Class: introns
  Correct:   132
  Incorrect: 74
  Rejected:  0
  Misclassified as:
    exons: 39
    repeats: 13
    promoters: 22

Class: repeats
  Correct:   44
  Incorrect: 137
  Rejected:  0
  Misclassified as:
    introns: 79
    exons: 25
    promoters: 33

===== PERFORMANCE FOR k = 2 =====

Class: promoters
  Correct:   265
  Incorrect: 67
  Rejected:  8
  Misclassified as:
    introns: 18
    repeats: 10
    exons: 39

Class: exons
  Correct:   165
  Incorrect: 150
  Rejected:  1
  Misclassified as:
    promoters: 71
    repeats: 55
    introns: 24

Class: introns
  Correct:   135
  Incorrect: 71
  Rejected:  0
  Misclassified as:
    exons: 45
    promoters: 10
    repeats: 16

Cl

### Preliminary Analysis of Classification Results from Final Run

Across all six Markov orders, the classifier performs strongly for promoters, introns, and—at higher k—exons, demonstrating that the Markov chain framework is functioning exactly as intended for canonical genomic classes. Promoters show the highest accuracy across all k values, with correct classifications rising from 250 at k=1 to 282 at k=6. This reflects the fact that promoters contain dense, information‑rich local motifs (e.g., TATA‑like elements, CpG enrichment, transcription factor binding clusters) that are well captured by short‑range k‑mer statistics. Even at low k, these motifs create distinctive transition patterns that separate promoters from the other classes. The misclassification patterns reinforce this: at k=1, promoters are most often misclassified as exons (60 cases) and introns (18 cases). This is biologically sensible—exons share GC‑rich regions with promoters, and introns share AT‑rich stretches—so when the promoter signal is weak or the sequence is atypical, the classifier defaults to the nearest neighbor in k‑mer space.

Exons show a clear improvement with increasing k, rising from 128 correct at k=1 to 213 correct at k=6. This trend is exactly what we expect from a Markov model: higher‑order k‑mers capture codon structure, 3‑base periodicity, and splice‑proximal motifs that are invisible at lower orders. The misclassification pattern also makes biological sense. At k=1, exons are most often misclassified as promoters (100 cases), because both classes share GC‑rich regions and structured motifs. As k increases, exon→promoter errors drop to 40 at k=6, while exon→repeat errors remain relatively stable (41 at k=6), reflecting the fact that some exons—especially short or low‑complexity ones—lack strong coding signatures and drift toward the more entropic classes.

Introns remain consistently well‑classified across all k values, with correct counts ranging from 132 at k=1 to 155 at k=6. Introns form a relatively homogeneous class with characteristic base composition and weak but consistent structural signals. Their misclassification patterns are stable and biologically intuitive: introns are most often misclassified as exons (22–45 cases depending on k), reflecting the shared splice‑adjacent regions and occasional GC‑rich segments. The fact that introns rarely get misclassified as promoters or repeats further confirms that the Markov models are capturing the broad compositional differences between these classes.

The repeats class, however, performs poorly across all k values, with correct classifications ranging only from 42–63 across k=1–6. This is not a coding error; it is a direct reflection of the biological and statistical properties of repetitive DNA. Unlike promoters, exons, or introns, “repeats” is not a single coherent sequence class. It is a heterogeneous mixture of LINEs, SINEs, LTRs, microsatellites, low‑complexity regions, and other transposable element families—each with distinct consensus motifs, GC content, evolutionary age, and mutation patterns. A single Markov model cannot represent this diversity. The model is forced to average over incompatible sequence distributions, producing a diffuse, low‑information transition matrix that cannot compete with the more coherent promoter, exon, and intron models. This is why repeats are overwhelmingly misclassified as introns (70–82 cases per k): both classes share long unstructured regions, AT‑rich stretches, and weak positional constraints, making introns the “nearest neighbor” when the repeat model lacks discriminative power.

Taken together, these results illustrate both the strengths and the limitations of Markov chain classifiers in genomics. For classes with coherent, motif‑driven sequence structure—promoters, exons, introns—the Markov framework is highly effective and improves predictably with increasing k. For heterogeneous or structurally complex classes like repeats, however, the assumptions of fixed‑order Markov models break down. The classifier’s behavior across k=1–6 is therefore not only successful but also deeply diagnostic: it reveals where the model excels, where it fails, and why. This evaluation provides a clear roadmap for future improvements, such as training family‑specific repeat models, increasing repeat training data, or adopting models capable of capturing long‑range structure (e.g., HMMs or neural sequence models).

In [13]:
from data_preparation import count_seqs_in_fasta, total_bp_in_fasta

filepath_test_promoters = "data/final/test/promoters_test.fa"
filepath_test_exons = "data/final/test/exons_test.fa"
filepath_test_introns = "data/final/test/introns_test.fa"
filepath_test_repeats = "data/final/test/repeats_test.fa"

num_seqs_promoters_test = count_seqs_in_fasta(filepath_test_promoters)
num_seqs_exons_test = count_seqs_in_fasta(filepath_test_exons)
num_seqs_introns_test = count_seqs_in_fasta(filepath_test_introns)
num_seqs_repeats_test = count_seqs_in_fasta(filepath_test_repeats)
num_bp_promoters_test = total_bp_in_fasta(filepath_test_promoters)
num_bp_exons_test = total_bp_in_fasta(filepath_test_exons)
num_bp_introns_test = total_bp_in_fasta(filepath_test_introns)
num_bp_repeats_test = total_bp_in_fasta(filepath_test_repeats)

print(f"Number of sequences in test promoters: {num_seqs_promoters_test}")
print(f"Number of bp in test promoters: {num_bp_promoters_test}")
print(f"Number of sequences in test exons: {num_seqs_exons_test}")
print(f"Number of bp in test exons: {num_bp_exons_test}")
print(f"Number of sequences in test introns: {num_seqs_introns_test}")
print(f"Number of bp in test introns: {num_bp_introns_test}")
print(f"Number of sequences in test repeats: {num_seqs_repeats_test}")
print(f"Number of bp in test repeats: {num_bp_repeats_test}")

Number of sequences in test promoters: 340
Number of bp in test promoters: 170340
Number of sequences in test exons: 314
Number of bp in test exons: 331749
Number of sequences in test introns: 206
Number of bp in test introns: 512046
Number of sequences in test repeats: 189
Number of bp in test repeats: 55799


| k | Promoters (82.9% max)  | Exons (67.8% max)   | Introns (75.2% max)   | Repeats (33.3% max)  |
|---|------------------------|---------------------|-----------------------|----------------------|
| 1 | 73.5% (250/340)        | 40.8% (128/314)     | 64.1% (132/206)       | 23.3% (44/189)       |
| 2 | 77.9% (265/340)        | 52.5% (165/314)     | 65.5% (135/206)       | 22.2% (42/189)       |
| 3 | 80.0% (272/340)        | 57.6% (181/314)     | 68.0% (140/206)       | 25.9% (49/189)       |
| 4 | 81.8% (278/340)        | 60.8% (191/314)     | 71.4% (147/206)       | 28.6% (54/189)       |
| 5 | 82.1% (279/340)        | 64.0% (201/314)     | 74.3% (153/206)       | 31.7% (60/189)       |
| 6 | 82.9% (282/340)        | 67.8% (213/314)     | 75.2% (155/206)       | 33.3% (63/189)       |



### Calculate Per-k and Per-Class Results

In [9]:
def evaluate_by_k(results_by_k, class_labels):
    """
    Compute accuracy and class metrics for each k value.
    Parameters:
        results_by_k (dict[int, list[dict]]): mapping k → list of result dicts
        class_labels (list[str]): ordered list of class names
    Returns:
        dict[int, dict]: mapping k → metrics including confusion matrix,
                         overall accuracy, per-class accuracy, and class metrics
    """
    summary = {}

    for k, results in results_by_k.items():
        # Convert results into (true, pred, seq) tuples for existing functions
        formatted = [(r["true_label"], r["predicted_label"], None) for r in results]

        conf = compute_confusion_matrix(formatted, class_labels)
        overall = compute_overall_accuracy(conf)
        per_class_acc = compute_per_class_accuracy(conf, class_labels)
        class_metrics = compute_class_metrics(conf, class_labels)

        summary[k] = {
            "confusion_matrix": conf,
            "overall_accuracy": overall,
            "per_class_accuracy": per_class_acc,
            "class_metrics": class_metrics
        }

    return summary
